# Baseline Model

In [14]:
# Load data
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

# Set up the inputs
df = pd.read_parquet("../data/shots_features_wc2022.parquet")

FEATURES = ["distance", "angle", "is_header"]
x = df[FEATURES].astype(float)
y = df.goal.astype(int)

In [15]:
# Split into training and test sets
x_train, x_test, y_train, y_test, sb_train, sb_test = train_test_split(x, y, df.shot_statsbomb_xg, test_size = 0.25, stratify = y, random_state = 42)

# Using logistic regression model as it gives an output between 0 and 1, which makes sense for goal probability
model = LogisticRegression()

# Model fits to find values that make its predicted probabilities match actual goals in training set as close as possible.
# Does this by minimizing log loss.
model.fit(x_train, y_train)

# Gives probability for each test shot
# Only shows the 'goal' value column
my_xg = model.predict_proba(x_test)[:, 1]

# Scoring Model against StatsBomb's on same Test Shots

In [16]:
def score(name, probs):
    # Returns info about overall statistics of xG model
    return{
        "model" : name,
        "log_loss" : log_loss(y_test, probs), 
        "brier" : brier_score_loss(y_test, probs), 
        "roc_auc" : roc_auc_score(y_test, probs), 
        "total_xG" : probs.sum(),
    }

# Compares results of my model vs StatsBomb's model
results = pd.DataFrame([score("My Model", my_xg), score("StatsBomb xG", sb_test)])
# Add in actual goals to results
results["actual_goals"] = y_test.sum()

results

,model,log_loss,brier,roc_auc,total_xG,actual_goals
0,My Model,0.278713,0.081834,0.806661,35.046636,38
1,StatsBomb xG,0.255205,0.074206,0.843997,33.457638,38


In [17]:
# What the model learned to interpret xG based on various factors
pd.Series(model.coef_[0], index = FEATURES)

# Covert log-odds to odds multipliers
# e.g. 0.89 for distance = each extra yard cuts odds of scoring by ~11%
np.exp(pd.Series(model.coef_[0], index = FEATURES))

distance     0.898644
angle        1.022236
is_header    0.281460
dtype: float64